#### 提取管腔面积最小的slice，并计算该slice的管腔面积、管壁面积、血管面积和标准化管壁指数

In [5]:
import os
import re
import cv2
import numpy as np
import SimpleITK as sitk
import pandas as pd

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    # 正则表达式匹配以 "slice_" 开头，后跟一个或多个数字，直到遇到下划线或点号
    match = re.search(r'slice_(\d+)[_.]', filename)
    # 如果匹配成功，返回切片编号的整数形式，否则返回None
    return int(match.group(1)) if match else None

def calculate_wall_area(wall_file):
    """通用函数计算管壁面积和检测内轮廓"""
    try:
        wall_img = sitk.ReadImage(wall_file)
        wall_data = sitk.GetArrayFromImage(wall_img)
        
        # 处理维度
        if wall_data.ndim == 3 and wall_data.shape[0] == 1:
            wall_data = wall_data[0]
        elif wall_data.ndim == 3 and wall_data.shape[0] > 1:
            raise ValueError("3D图像暂不支持")
        
        # 转换为OpenCV格式
        wall_binary = ((wall_data > 0) * 255).astype(np.uint8)
        
        # 检测轮廓
        contours, hierarchy = cv2.findContours(wall_binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"警告：管壁mask未检测到轮廓 {wall_file}")
            return None, None
        
        # 计算管壁面积
        wall_spacing = wall_img.GetSpacing()[:2]
        wall_voxel_area = wall_spacing[0] * wall_spacing[1]
        wall_area = np.sum(wall_binary > 0) * wall_voxel_area
        
        # 检测内轮廓
        hierarchy = hierarchy[0]
        has_inner_contour = any(h[3] != -1 for h in hierarchy)
        
        return wall_area, has_inner_contour
    except Exception as e:
        print(f"处理管壁文件失败 {wall_file}: {str(e)}")
        return None, None

def process_id_folder(lumen_case_path, wall_case_path, output_dir, case_id):
    """处理单个病例文件夹"""
    candidates = []
    lumen_files = [f for f in os.listdir(lumen_case_path) if f.endswith(".mhd")]

    for lumen_file in lumen_files:
        # 解析切片编号
        slice_idx = find_slice_number(lumen_file)
        if slice_idx is None:
            print(f"跳过无法解析的文件: {lumen_file}")
            continue

        try:
            # ==================== 处理管腔mask ====================
            lumen_path = os.path.join(lumen_case_path, lumen_file)
            lumen_img = sitk.ReadImage(lumen_path)
            lumen_data = sitk.GetArrayFromImage(lumen_img)
            
            # 处理维度
            if lumen_data.ndim == 3 and lumen_data.shape[0] == 1:
                lumen_data = lumen_data[0]
            elif lumen_data.ndim == 3 and lumen_data.shape[0] > 1:
                raise ValueError("3D图像暂不支持")
            
            # 计算管腔面积
            lumen_spacing = lumen_img.GetSpacing()[:2]
            voxel_area = lumen_spacing[0] * lumen_spacing[1]
            lumen_area = np.sum(lumen_data > 0) * voxel_area
        except Exception as e:
            print(f"处理管腔文件失败 {lumen_file}: {str(e)}")
            continue

        # ==================== 处理对应管壁文件 ====================
        # 查找对应的管壁文件
        wall_files = [f for f in os.listdir(wall_case_path) if f.endswith(".mhd")]
        if not wall_files:
            print(f"无管壁文件")
            continue
        for wall_file in wall_files:
            wall_file_path = os.path.join(wall_case_path,wall_file)
            wall_num = find_slice_number(wall_file)
            if wall_num == slice_idx:
                # 计算管壁面积（无论管腔面积如何）
                wall_area, has_inner_contour = calculate_wall_area(wall_file_path)
                if wall_area is None:
                    print("管壁面积为0！")
                    continue  # 跳过无效管壁文件

                # ==================== 处理不同情况 ====================
                if lumen_area == 0:
                    if has_inner_contour:
                        print(f"异常：切片 {lumen_file} 管腔面积为0但存在内轮廓")
                        continue
                    candidate_type = 'zero_valid'
                else:
                    candidate_type = 'normal'

                candidates.append({
                    'type': candidate_type,
                    'lumen_area': lumen_area,
                    'wall_area': wall_area,
                    'filename': lumen_file,
                    'wall_file':wall_file,
                    'slice_idx': slice_idx
                })

    # ==================== 候选切片处理 ====================
    if not candidates:
        print(f"没有有效切片: {lumen_case_path}")
        return

    # 选择最小管腔面积的候选
    min_lumen_area = min(c['lumen_area'] for c in candidates)
    min_candidates = [c for c in candidates if c['lumen_area'] == min_lumen_area]

    final_candidate = None
    if min_lumen_area > 0:
        # 新的选择策略：相同最小管腔面积时选管壁最大的
        if len(min_candidates) > 1:
            print(f"注意：发现 {len(min_candidates)} 个最小管腔面积相同的切片（{min_lumen_area:.2f} mm²），选择管壁面积最大的")
        final_candidate = max(min_candidates, key=lambda x: x['wall_area'])
    else:
        # 保持原有零面积处理逻辑
        if len(min_candidates) > 1:
            print(f"注意：发现 {len(min_candidates)} 个有效零面积切片，选择管壁面积最大的")
        final_candidate = max(min_candidates, key=lambda x: x['wall_area'])

    # ==================== 保存结果 ====================
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # 保存管腔mask（原始文件名）
        lumen_src = os.path.join(lumen_case_path, final_candidate['filename'])
        lumen_dst = os.path.join(output_dir, final_candidate['filename'])
        sitk.WriteImage(sitk.ReadImage(lumen_src), lumen_dst)
        
        # 保存管壁mask（原始文件名）
        wall_src = os.path.join(wall_case_path, final_candidate['wall_file'])
        wall_dst = os.path.join(output_dir, final_candidate['wall_file'])
        sitk.WriteImage(sitk.ReadImage(wall_src), wall_dst)
        
        vessel_area = final_candidate['lumen_area'] + final_candidate['wall_area']
        nwi = final_candidate['wall_area'] / vessel_area if vessel_area != 0 else 0.0
        
        # 添加特征到全局列表
        feature_row = {
            'Case ID': case_id,
            'Slice Index': final_candidate['slice_idx'],
            'Lumen Area (mm²)': round(final_candidate['lumen_area'], 4),
            'Wall Area (mm²)': round(final_candidate['wall_area'], 4),
            'Vessel Area (mm²)': round(vessel_area, 4),
            'Normalized Wall Index': round(nwi, 4)
        }
        
        #all_features.append(feature_row)
        print(f"处理完成：{case_id}")
        return feature_row
    
    except Exception as e:
        print(f"保存失败: {str(e)}")
        return 0

lumen_mask_path = "PROJECT_ROOT/lumen"
wall_mask_path = "PROJECT_ROOT/wall"  
output_base = "PROJECT_ROOT/T1_MinLumen_slice"
excel_path = "PROJECT_ROOT/feature_excel"

if __name__ == "__main__":
    all_features = []
    # 遍历所有病例
    for case_id in os.listdir(lumen_mask_path):
        lumen_case_path = os.path.join(lumen_mask_path, case_id)
        wall_case_path = os.path.join(wall_mask_path, case_id)
        output_dir = os.path.join(output_base, case_id)

        if not os.path.isdir(lumen_case_path):
            print(f"缺失管腔病例目录: {lumen_case_path}")
            continue
        if not os.path.isdir(wall_case_path):
            print(f"缺失管壁病例目录: {wall_case_path}")
            continue

        features = process_id_folder(lumen_case_path, wall_case_path, output_dir, case_id)
        all_features.append(features)

    if all_features:
        # 保存路径
        df = pd.DataFrame(all_features)
        excel_path = os.path.join(excel_path, "1.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"\n所有数据已保存到: {excel_path}")
        print(f"总病例数: {len(df)}")
    else:
        print("\n警告：未找到任何有效病例数据")
    
    print("\n处理完成! 输出目录:", output_base)

处理完成：qijundong_0012274330
处理完成：luzhengqing_0009781945
处理完成：liulanxiang_0011602026
处理完成：caohuiqi_0012455220
处理完成：chenzhiming_916310
处理完成：fangduanyun_0010315958
处理完成：xingjingguo_0011791960
处理完成：zhaoya_0009441311
处理完成：zhangguifang1_0011552299
处理完成：wanglingxiang_0010416096
注意：发现 2 个最小管腔面积相同的切片（0.21 mm²），选择管壁面积最大的
处理完成：miyuhua_0006554766
处理完成：fangyude_0007895914
处理完成：jiahonglong_0011122136
处理完成：wangzhaosong_0006229548
处理完成：zhangchaoping_0009307570
处理完成：peixufeng_0010408826
处理完成：jinguisheng_990843
注意：发现 3 个有效零面积切片，选择管壁面积最大的
处理完成：lichangyong_0008651509
处理完成：zhangjingyi_0007670576
处理完成：geshaogui_0010116924
处理完成：wangcuilan_0010627868
处理完成：sunguibao_0009753094
处理完成：zhouguangde_0010412552
处理完成：liushuluan_0012292935
处理完成：huangyumei_0007983670
处理完成：chenjingle_0010828341
处理完成：zhangwei_0009159524
处理完成：chenjun_0009773077
处理完成：wangzibing_0010678758
处理完成：chenfengzhen_2443243
处理完成：chengjianchun_0011669206
注意：发现 2 个最小管腔面积相同的切片（0.12 mm²），选择管壁面积最大的
处理完成：jiajianshe_0010668284
处理完成：gexiangmei_0008254165
处理完成：

#### 匹配最小管腔切片mask对应的原图

In [ ]:
import os
import shutil
import re
#正式路径
# 源路径（包含筛选出的min_slice编号）
minlumen_base = r"PROJECT_ROOT/T1_MinLumen_slice"
# 待匹配的切片源路径
source_base = r"PROJECT_ROOT/T1_slice"
# 目标保存路径
target_base = r"PROJECT_ROOT/T1_MinLumen_slice"

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    # 正则表达式匹配以 "slice_" 开头，后跟一个或多个数字，直到遇到下划线或点号
    match = re.search(r'slice[_]?(\d+)', filename)
    # 如果匹配成功，返回切片编号的整数形式，否则返回None
    return int(match.group(1))

def copy_matching_slices(num):
    #num = 0
    # 遍历MinLumen目录结构
    for id_dir in os.listdir(minlumen_base):
        minlumen_path = os.path.join(minlumen_base,id_dir)    
        print(f"⚪⚪处理患者 {id_dir}")
        # 在每个目录中查找min_slice文件
        for file in os.listdir(minlumen_path):
            if file.endswith("lumen.mhd"):    
                # 提取切片编号
                slice_num = find_slice_number(file)
                #print(slice_num)
                #切片编号有可能为0,所以这个判断会筛掉为0的切片
                # if not slice_num:
                #     continue
                    
                print(f"处理患者 {id_dir} 的切片 {slice_num}")

                # 构建源文件路径模板（根据实际文件名格式调整）
                #source_pattern = f"*_slice_{slice_num}.mhd"
                source_dir = os.path.join(source_base, id_dir)

                # 查找匹配的源文件
                found_files = []
                if os.path.exists(source_dir):
                    for src_file in os.listdir(source_dir):
                        src_num = find_slice_number(src_file)
                        #print(f"遍历到切片{src_num}")
                        if src_num == slice_num and src_file.endswith('.mhd'):
                            found_files.append(src_file)
                            # 同时添加对应的.raw文件
                            raw_file = src_file.replace(".mhd", ".raw")
                            if os.path.exists(os.path.join(source_dir, raw_file)):
                                found_files.append(raw_file)
                            print(f"✅成功找到{src_num}切片")
                        continue
                
                # 创建目标目录
                target_dir = os.path.join(target_base, id_dir)
                os.makedirs(target_dir, exist_ok=True)

                # 复制文件
                if found_files:
                    for f in found_files:
                        src_path = os.path.join(source_dir, f)
                        dst_path = os.path.join(target_dir, f)
                        shutil.copy2(src_path, dst_path)
                        print(f"已复制: {f} -> {dst_path}")
                else:
                    print(f"警告: 未找到患者 {id_dir} 切片 {slice_num} 的匹配文件")
        num+=1
    print(f"共处理{num}个数据")

if __name__ == "__main__":
    num =0
    copy_matching_slices(num)
    print("处理完成！所有匹配切片已保存到：", target_base)

#### 直径、厚度相关特征(根据最小管腔面积切片计算)

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion

def visualize_masks(image_path, mask_path, center_x, center_y):
    """可视化图像和mask轮廓，并在图像上显示中心点"""
    image = sitk.GetArrayFromImage(sitk.ReadImage(image_path))
    image = np.squeeze(image)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    mask = np.squeeze(mask)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='r', linewidths=0.5)
    # 在图像上绘制中心点，'ro'表示红色的点
    plt.plot(center_x, center_y, 'go',markersize=1)  # 红色点ro
    plt.title('Image with Mask and Center Point')
    
    plt.show()
    plt.close()


def calculate_lumen_diameters(original_file_path,mask_file_path,id_dir):
    """
    计算经过调整中心的管腔直径（最大、最小、平均）
    """
    # 读取并预处理图像
    mask_image = sitk.ReadImage(mask_file_path)
    mask = sitk.GetArrayFromImage(mask_image)
    # 处理三维数据（增加维度检查）
    if mask.ndim == 3:
        # 确保选择正确的切片
        if mask.shape[0] > 1:
            mask = mask[0, :, :]  # 取第一个切片
        else:
            mask = np.squeeze(mask, axis=0)  # 去除单例维度
    mask = mask.astype(np.uint8)
    
    img_spacing = mask_image.GetSpacing()[:2]
    spacing_x = img_spacing[0]
    spacing_y = img_spacing[1]
    
    # 检查是否存在管腔
    if np.sum(mask) == 0:
        print(f"{id_dir}🚫❌管腔不存在")
        return 0.0, 0.0, 0.0
    
    # 智能中心点调整 ====================================================
    # 初始中心
    height, width = mask.shape
    initial_center = (width // 2, height // 2)
    
    # 找到最大连通域
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0, 0.0, 0.0
    
    # 选择最大轮廓（并验证维度）
    main_contour = max(contours, key=cv2.contourArea)
    
    # 检查初始中心状态（带符号距离）
    center_status = cv2.pointPolygonTest(main_contour, initial_center, True)
    
    # 动态调整中心（包含在轮廓上的情况）
    if center_status <= 0:  # 在外部或边界上
        # 三级调整策略
        # 1. 质心法
        M = cv2.moments(main_contour)
        if M["m00"] > 0:
            new_center = (int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"]))
            # 严格验证新中心是否在内部
            if cv2.pointPolygonTest(main_contour, new_center, False) <= 0:
                # 2. 距离变换法
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)
                _, _, _, new_center = cv2.minMaxLoc(dist_map)
        else:
            new_center = initial_center
    else:
        new_center = initial_center
    
    center_x, center_y = new_center
    
    # 可视化验证 ======================================================
    # visualize_masks(original_file_path,mask_file_path,center_x, center_y)
    
    # 计算各方向最大半径 ==============================================
    max_radius = np.zeros(360)
    
    # 修正轮廓点访问方式
    if len(main_contour.shape) == 3:  # 确保轮廓是(N,1,2)格式
        contour_points = main_contour.squeeze(axis=1)  # 转换为(N,2)
    else:
        contour_points = main_contour
    
    for point in contour_points:
        x, y = point
        dx = (x - center_x)*spacing_x
        dy = (y - center_y)*spacing_y
        r = np.hypot(dx, dy)
        theta_deg = np.degrees(np.arctan2(dy, dx)) % 360
        theta_idx = int(round(theta_deg))
        if theta_idx >= 360: 
            theta_idx = 0
        if r > max_radius[theta_idx]:
            max_radius[theta_idx] = r
    
    # 计算有效直径 ==================================================
    diameters = []
    for theta in range(180):  # 利用对称性优化计算
        opposite_theta = (theta + 180) % 360
        if max_radius[theta] > 0 or max_radius[opposite_theta] > 0:
            diameters.append(max_radius[theta] + max_radius[opposite_theta])
    # 处理无有效直径的情况
    if not diameters:
        print(f"{id_dir}🚫❌直径不存在！")
        return 0, 0, 0
    
    max_diameter = max(diameters)
    min_diameter = min(diameters)
    avg_diameter = np.mean(diameters)
    
    return max_diameter, min_diameter, avg_diameter


# 管壁管腔mask都在这条路径里面
mask_path = "PROJECT_ROOT/T1_MinLumen_slice"
#特征数据保存路径
output_path = "PROJECT_ROOT/feature_excel"
# 定义结果保存路径
output_excel = os.path.join(output_path, "2.xlsx")

def calculate_metrics(original_file_path,lumen_mask_path, wall_mask_path, id_dir):
    # 管腔直径计算
    lumen_max_d,lumen_min_d,lumen_avg_d = calculate_lumen_diameters(original_file_path,lumen_mask_path,id_dir)    
    
    return {
        'ID': id_dir,
        'lumen_diameter_max_slice(mm)': lumen_max_d,
        'lumen_diameter_min_slice(mm)': lumen_min_d,
        'lumen_diameter_avg_slice(mm)': lumen_avg_d,
    }

results = []
for id_dir in os.listdir(mask_path):
    files_path = os.path.join(mask_path, id_dir)

    original_file = [f for f in os.listdir(files_path) if f.endswith('.mhd') and '_lumen' not in f and '_wall' not in f]
    lumen_file = [f for f in os.listdir(files_path) if f.endswith('lumen.mhd')]
    wall_file = [f for f in os.listdir(files_path) if f.endswith('wall.mhd')]

    # 两个文件都存在
    if lumen_file and wall_file and original_file:
        original_file_path = os.path.join(files_path, original_file[0])
        lumen_file_path = os.path.join(files_path, lumen_file[0])
        wall_file_path = os.path.join(files_path, wall_file[0])
        
        # 计算指标
        try:
            metrics = calculate_metrics(original_file_path,lumen_file_path,wall_file_path,id_dir)  # 使用文件名作为ID
            results.append(metrics)
            print(f"成功处理: {id_dir}")
        except Exception as e:
            print(f"处理失败 {id_dir}: {e}")  # 打印异常信息
    else:
        print(f"{id_dir}可能缺少管腔或管壁.mhd文件")

# 保存结果为excel文件
if results:
    df = pd.DataFrame(results)
    with pd.ExcelWriter(output_excel,engine='openpyxl') as writer:
        df.to_excel(writer,index=False,sheet_name="diameter Features")
    # 按ID排序
    print(f"结果已保存至: {output_excel}")
    print(f"共处理 {len(results)} 个样本")
else:
    print("未找到可处理的数据")

In [ ]:
import numpy as np
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
import os
import pandas as pd

def visualize_masks(image_path, mask_path, center_x, center_y):
    """可视化图像和mask轮廓，并在图像上显示中心点"""
    image = sitk.GetArrayFromImage(sitk.ReadImage(image_path))
    image = np.squeeze(image)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    mask = np.squeeze(mask)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='r', linewidths=0.5)
    # 在图像上绘制中心点，'ro'表示红色的点
    plt.plot(center_x, center_y, 'go',markersize=1)  # 红色点ro
    plt.title('Image with Mask and Center Point')
    plt.show()
    plt.close()

def calculate_wall_diameters(original_file_path,wall_file_path,id_dir):
    # 读取 .mhd 文件
    image = sitk.ReadImage(wall_file_path)
    mask = sitk.GetArrayFromImage(image)
    
    img_spacing = image.GetSpacing()[:2]
    spacing_x = img_spacing[0]
    spacing_y = img_spacing[1]
    
    # 处理三维数据，选择有效切片
    if mask.ndim == 3:
        slice_areas = [np.count_nonzero(slice) for slice in mask]
        chosen_slice = np.argmax(slice_areas)
        mask = mask[chosen_slice, :, :]
    
    # 转换为8位灰度图并进行二值化
    mask = mask.astype(np.uint8)
    _, mask_bin = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY)
    
    height, width = mask_bin.shape

    # 检测外轮廓
    contours_wall, hierarchy = cv2.findContours(mask_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    #如果出现没有管壁的情况，这里可能会报错，目前没有好像！！！！！！！！！！（3D报错了，这里暂时不改）
    if not contours_wall:
        raise ValueError(f"{id_dir}🚫❌未检测到管壁轮廓")
    
    # 分离外轮廓和内轮廓
    outer_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] == -1]  # 外轮廓列表
    inner_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] != -1]  # 内轮廓列表
    
    # 检查外轮廓存在性
    if not outer_contours:
        raise ValueError(f"{id_dir}🚫❌未检测到管壁外边界")  # 如果没有外轮廓，抛出异常
    else:
        largest_outer = max(outer_contours, key=cv2.contourArea)
    
    # 智能中心点调整 ====================================================
    initial_center = (width//2, height//2)  # 初始中心点为图像中心
    center_x, center_y = initial_center
    
    # 存在内轮廓时的处理
    if inner_contours:
        #print(f"{id_dir}✔️直接使用内轮廓计算")
        # 只处理最大内轮廓
        largest_inner = max(inner_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_inner, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_inner)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_inner, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_inner)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_inner, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以内轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算(内轮廓)")
    elif outer_contours and not inner_contours:
        print(f"{id_dir}✔️未检测到内轮廓,直接使用外轮廓计算")
         # 只处理最大内轮廓
        # largest_outer = max(outer_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_outer, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_outer)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_outer, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_outer)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_outer, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以外轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算")
        
    else:
        print(f"{id_dir}🚫❌内轮廓和外轮廓都未检测到！")
    
    #可视化
    # visualize_masks(original_file_path, wall_file_path, center_x, center_y)

    # 投影法计算各方向最大半径
    max_radius = np.zeros(360)
    for theta in range(360):
        theta_rad = np.radians(theta)
        direction = np.array([np.cos(theta_rad), np.sin(theta_rad)])
        max_proj = -np.inf
        
        for point in largest_outer:
            x, y = point[0]
            dx = (x - center_x)*spacing_x
            dy = (y - center_y)*spacing_y
            proj = dx*direction[0] + dy*direction[1]
            if proj > max_proj:
                max_proj = proj
        
        if max_proj > 0:
            max_radius[theta] = max_proj  # 方向向量已单位化

    # 计算有效直径
    diameters = []
    for theta in range(180):  # 只需遍历前180度
        opposite_theta = theta + 180
        if max_radius[theta] > 0 and max_radius[opposite_theta] > 0:
            diameters.append(max_radius[theta] + max_radius[opposite_theta])
    
    if not diameters:
        print(f"{id_dir}🚫❌直径计算出错")
        return 0.0, 0.0, 0.0
    
    return np.max(diameters), np.min(diameters), np.mean(diameters)

# 管壁管腔mask都在这条路径里面
mask_path = "PROJECT_ROOT/T1_MinLumen_slice"
#特征数据保存路径
output_path = "PROJECT_ROOT/feature_excel"
# 定义结果保存路径
output_excel = os.path.join(output_path, "3.xlsx")

def calculate_metrics(original_file_path,lumen_mask_path, wall_mask_path, id_dir):
    # 管壁直径计算
    wall_max_d, wall_min_d,wall_avg_d= calculate_wall_diameters(original_file_path,wall_mask_path,id_dir)      
    
    return {
        'ID': id_dir,
        'wall_diameter_max_slice(mm)': wall_max_d,
        'wall_diameter_min_slice(mm)': wall_min_d,
        'wall_diameter_avg_slice(mm)': wall_avg_d
    }

results = []
for id_dir in os.listdir(mask_path):
    print(f"⚪正在处理{id_dir}")
    files_path = os.path.join(mask_path, id_dir)

    original_file = [f for f in os.listdir(files_path) if f.endswith('.mhd') and '_lumen' not in f and '_wall' not in f]
    lumen_file = [f for f in os.listdir(files_path) if f.endswith('lumen.mhd')]
    wall_file = [f for f in os.listdir(files_path) if f.endswith('wall.mhd')]

    # 两个文件都存在
    if lumen_file and wall_file and original_file:
        original_file_path = os.path.join(files_path, original_file[0])
        lumen_file_path = os.path.join(files_path, lumen_file[0])
        wall_file_path = os.path.join(files_path, wall_file[0])
        
        # 计算指标
        try:
            metrics = calculate_metrics(original_file_path,lumen_file_path,wall_file_path,id_dir)  # 使用文件名作为ID
            results.append(metrics)
            print(f"✅成功处理: {id_dir}")
        except Exception as e:
            print(f"❌处理失败 {id_dir}: {e}")  # 打印异常信息
    else:
        print(f"⚠️{id_dir}可能缺少管腔或管壁.mhd文件")
    print('-'*60)

# 保存结果为excel文件
if results:
    df = pd.DataFrame(results)
    with pd.ExcelWriter(output_excel,engine='openpyxl') as writer:
        df.to_excel(writer,index=False,sheet_name="diameter Features")
    # 按ID排序
    print(f"结果已保存至: {output_excel}")
    print(f"共处理 {len(results)} 个样本")
else:
    print("未找到可处理的数据")

#### 适用于求管壁厚度

In [ ]:
import numpy as np
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def visualize_masks(image_path, mask_path, center_x, center_y):
    """可视化图像和mask轮廓，并在图像上显示中心点"""
    image = sitk.GetArrayFromImage(sitk.ReadImage(image_path))
    image = np.squeeze(image)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    mask = np.squeeze(mask)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='r', linewidths=0.5)
    # 在图像上绘制中心点，'ro'表示红色的点
    plt.plot(center_x, center_y, 'go',markersize=1)  # 红色点ro
    plt.title('Image with Mask and Center Point')
    plt.show()
    plt.close()
    
# 改进的半径计算 ====================================================
def process_contour(contours, img_spacing,center, is_outer=True):
    radius_array = np.zeros(360) if is_outer else np.full(360, np.inf)
    all_points = []
    
    spacing_x = img_spacing[0]
    spacing_y = img_spacing[1]
    
    for contour in contours:
        points = contour.squeeze()
        if points.ndim != 2 or points.shape[1] != 2:
            continue
        all_points.extend(points.tolist())
    
    if not all_points:
        return radius_array
    
    # 转换为极坐标
    points_array = np.array(all_points)
    dx = (points_array[:, 0] - center[0])*spacing_x
    dy = (points_array[:, 1] - center[1])*spacing_y
    radii = np.hypot(dx, dy)
    thetas = np.degrees(np.arctan2(dy, dx)) % 360

    # 改进的角度分配策略：分配到最近的三个角度
    for theta, r in zip(thetas, radii):
        base_angle = int(np.round(theta))
        for offset in [-1, 0, 1]:  # 覆盖三个相邻角度
            angle = (base_angle + offset) % 360
            if is_outer:
                if r > radius_array[angle]:
                    radius_array[angle] = r
            else:
                if r < radius_array[angle]:
                    radius_array[angle] = r
    return radius_array

def calculate_wall_thickness(original_path,mask_path,id_dir):
    
    # 读取 .mhd 文件
    image = sitk.ReadImage(mask_path)
    mask = sitk.GetArrayFromImage(image)
    
    img_spacing = image.GetSpacing()[:2]
    # spacing_x = img_spacing[0]
    # spacing_y = img_spacing[1]
    
    # original_img = sitk.ReadImage(original_path)
    # original = sitk.GetArrayFromImage(original_img)
    
    # 处理三维数据
    if mask.ndim == 3:
        mask = mask[0, :, :]  # 如果是三维数据，只取第一个切片
    mask = mask.astype(np.uint8)  # 将数据类型转换为无符号8位整型
    #进行二值化
    _, mask_bin = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY)
    
    height, width = mask.shape  # 获取图像的高度和宽度
    
    # 检测轮廓
    contours_wall, hierarchy = cv2.findContours(mask_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if not contours_wall:
        print(f"❌❌{id_dir}未检测到任何轮廓")
        return 0,0,0
        #raise ValueError("未检测到任何轮廓")  # 如果没有检测到轮廓，抛出异常
    
    # 分离外轮廓和内轮廓
    outer_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] == -1]  # 外轮廓列表
    inner_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] != -1]  # 内轮廓列表
    
    # 检查外轮廓存在性
    if not outer_contours:
        #没有外轮廓，肯定没有内轮廓，应该是没有标签，为空
        print(f"❌❌{id_dir}未检测到任何外轮廓")
        return 0,0,0
        #raise ValueError(f"{id_dir}未检测到管壁外边界")  # 如果没有外轮廓，抛出异常
    else:
        largest_outer = max(outer_contours, key=cv2.contourArea)

    # 智能中心点调整 ====================================================
    initial_center = (width//2, height//2)  # 初始中心点为图像中心
    center_x, center_y = initial_center
    
    # 存在内轮廓时的处理
    if inner_contours:
        # 只处理最大内轮廓
        largest_inner = max(inner_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_inner, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_inner)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_inner, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_inner)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_inner, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"✔️{id_dir}以内轮廓为基准修改了中心")
        else:
            print(f"✔️{id_dir}使用原中心计算(内轮廓)")
    elif outer_contours:
        print(f"✔️{id_dir}未检测到内轮廓,直接使用外轮廓计算")
         # 只处理最大内轮廓
        #largest_outer = max(outer_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_outer, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_outer)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_outer, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_outer)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_outer, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"✔️{id_dir}以外轮廓为基准修改了中心")
        else:
            print(f"✔️{id_dir}使用原中心计算(外轮廓)")
    else:
        print(f"⚠️{id_dir}内轮廓和外轮廓都未检测到！")

    # 可视化 ==========================================================
    # visualize_masks(original_path, mask_path, center_x, center_y)
    
    # 计算半径 ========================================================
    #投影法计算个方向最大管壁厚度
    # 处理外轮廓
    #前面已经判断了，如果没有外轮廓，到不了这一步的，这里不用判断了
    max_radius = process_contour(outer_contours,img_spacing, (center_x, center_y), is_outer=True)
    
    # 处理内轮廓（智能插值）
    if inner_contours:
        # 初步处理内轮廓
        min_radius = process_contour(inner_contours, img_spacing,(center_x, center_y), is_outer=False)
        
        # 创建有效数据点集合
        valid_mask = min_radius != np.inf
        valid_angles = np.where(valid_mask)[0]
        valid_values = min_radius[valid_mask]
        
        # 智能插值策略（仅在外轮廓有效区域）
        outer_mask = max_radius > 1e-6
        if len(valid_angles) > 10:
            # 扩展角度处理周期边界
            extended_angles = np.concatenate([valid_angles-360, valid_angles, valid_angles+360])
            extended_values = np.concatenate([valid_values, valid_values, valid_values])
            
            # 创建插值函数
            f = interp1d(extended_angles, extended_values, 
                        kind='linear', bounds_error=False)
            
            # 仅在外轮廓有效区域插值
            need_interp = np.where(outer_mask & ~valid_mask)[0]
            if len(need_interp) > 0:
                interpolated = f(need_interp)
                interpolated = np.clip(interpolated, 0, max_radius[need_interp]*0.95)
                min_radius[need_interp] = interpolated
    else:
        #内轮廓的半径用min_radius表示，外轮廓半径用max_radius表示
        #如果没有内轮廓，则管壁厚度就是外轮廓的半径
        min_radius = np.zeros_like(max_radius)
        print(f"{id_dir}没有内轮廓，直接用外轮廓半径作为管壁厚度")

    # 改进的壁厚计算 ====================================================
    wall_thickness = []
    for theta in range(360):
        outer_r = max_radius[theta]
        if outer_r <= 1e-6:
            continue
            
        if inner_contours:
            inner_r = min_radius[theta]
            
            # 异常值处理策略
            if inner_r >= outer_r:
                print(f"❌❌❌❌❌❌❌{id_dir}内轮廓半径大于外轮廓半径！不可能呀，哥们，检查一下图像吧")
                # 三级修复策略
                # 1. 寻找相邻有效值
                neighbor_values = []
                for offset in [-2, -1, 1, 2]:
                    neighbor_theta = (theta + offset) % 360
                    if min_radius[neighbor_theta] < outer_r:
                        neighbor_values.append(min_radius[neighbor_theta])
                
                # 2. 使用统计值
                if neighbor_values:
                    inner_r = np.mean(neighbor_values)
                else:
                    # 3. 安全回退策略
                    inner_r = outer_r * 0.8
                
                # 最终保障
                inner_r = min(inner_r, outer_r*0.95)
            
            thickness = outer_r - inner_r
            wall_thickness.append(thickness)
        else:
            wall_thickness.append(outer_r)
    
    #这里考虑到了没有标签的情况，输出000        
    if wall_thickness:
        thickness_max_d = max(wall_thickness)  # 计算最大直径
        thickness_min_d = min(wall_thickness)  # 计算最小直径
        thickness_avg_d = np.mean(wall_thickness)  #计算平均直径
        return thickness_max_d,thickness_min_d,thickness_avg_d
    else:
        print(f"{id_dir}❌❌没有分割啥东西出来哦")
        return 0.0,0.0,0.0

# 管壁管腔mask都在这条路径里面
mask_path = "PROJECT_ROOT/T1_MinLumen_slice"
#特征数据保存路径
output_path = "PROJECT_ROOT/feature_excel"
# 定义结果保存路径
output_excel = os.path.join(output_path, "4.xlsx")

def calculate_metrics(original_file_path,lumen_mask_path, wall_mask_path, id_dir):
    # 管壁直径计算
   # 管壁厚度计算
    thickness_max_d,thickness_min_d,thickness_avg_d = calculate_wall_thickness(original_file_path,wall_mask_path,id_dir)   
    # 计算偏心指数
    if thickness_max_d:
        eccentricity_index = (thickness_max_d-thickness_min_d)/thickness_max_d
    else:
        print(f"{id_dir}管壁厚度为0")
        eccentricity_index= 0     
    
    return {
        'ID': id_dir,
        'wall_thickness_max_slice(mm)': thickness_max_d,
        'wall_thickness_min_slice(mm)': thickness_min_d,
        'wall_thickness_avg_slice(mm)': thickness_avg_d,
        'eccentricity_index_slice': eccentricity_index
    }

results = []
for id_dir in os.listdir(mask_path):
    print(f"⚪正在处理{id_dir}")
    files_path = os.path.join(mask_path, id_dir)

    original_file = [f for f in os.listdir(files_path) if f.endswith('.mhd') and '_lumen' not in f and '_wall' not in f]
    lumen_file = [f for f in os.listdir(files_path) if f.endswith('lumen.mhd')]
    wall_file = [f for f in os.listdir(files_path) if f.endswith('wall.mhd')]

    # 两个文件都存在
    if lumen_file and wall_file and original_file:
        original_file_path = os.path.join(files_path, original_file[0])
        lumen_file_path = os.path.join(files_path, lumen_file[0])
        wall_file_path = os.path.join(files_path, wall_file[0])
        
        # 计算指标
        try:
            metrics = calculate_metrics(original_file_path,lumen_file_path,wall_file_path,id_dir)  # 使用文件名作为ID
            results.append(metrics)
            print(f"✅成功处理: {id_dir}")
        except Exception as e:
            print(f"❌处理失败 {id_dir}: {e}")  # 打印异常信息
    else:
        print(f"⚠️{id_dir}可能缺少管腔或管壁或原始图像.mhd文件")
    print('-'*60)

# 保存结果为excel文件
if results:
    df = pd.DataFrame(results)
    with pd.ExcelWriter(output_excel,engine='openpyxl') as writer:
        df.to_excel(writer,index=False,sheet_name="thickness Features")
    # 按ID排序
    print(f"结果已保存至: {output_excel}")
    print(f"共处理 {len(results)} 个样本")
else:
    print("未找到可处理的数据")

#### 计算重构指数和狭窄率

In [3]:
import SimpleITK as sitk
import numpy as np
import os

def calculate_lumen_area(lumen_file):
    """
    计算管腔面积的函数

    参数:
    lumen_case_path (str): 管腔图像所在的文件夹路径
    lumen_file (str): 管腔图像文件名
    
    返回:
    float: 管腔的面积
    """
    try:
        # 读取管腔图像
        #lumen_path = os.path.join(lumen_case_path, lumen_file)
        lumen_img = sitk.ReadImage(lumen_file)
        lumen_data = sitk.GetArrayFromImage(lumen_img)
        
        # 处理维度
        if lumen_data.ndim == 3 and lumen_data.shape[0] == 1:
            lumen_data = lumen_data[0]
        elif lumen_data.ndim == 3 and lumen_data.shape[0] > 1:
            raise ValueError("3D图像暂不支持")
        
        # 计算管腔面积
        lumen_spacing = lumen_img.GetSpacing()[:2]
        voxel_area = lumen_spacing[0] * lumen_spacing[1]
        lumen_area = np.sum(lumen_data > 0) * voxel_area
        
        return lumen_area
    
    except Exception as e:
        print(f"处理管腔文件失败 {lumen_file}: {str(e)}")
        return None

def calculate_wall_area1(wall_file):
    """计算管壁面积的函数"""
    """计算管壁面积的函数（返回单个数值）"""
    try:
        # 读取图像
        wall_img = sitk.ReadImage(wall_file)
        wall_data = sitk.GetArrayFromImage(wall_img)
        
        # 处理维度
        if wall_data.ndim == 3:
            if wall_data.shape[0] == 1:
                wall_data = wall_data[0]
            elif wall_data.shape[0] > 1:
                # 取中间层作为代表
                mid_slice = wall_data.shape[0] // 2
                wall_data = wall_data[mid_slice]
        
        # 创建二值图像并计算面积
        wall_binary = (wall_data > 0).astype(np.uint8)
        wall_spacing = wall_img.GetSpacing()[:2]
        wall_voxel_area = wall_spacing[0] * wall_spacing[1]
        return np.sum(wall_binary) * wall_voxel_area  # 确保返回单个值
    
    except Exception as e:
        print(f"处理管壁文件失败 {wall_file}: {str(e)}")
        return 0  # 返回0避免后续计算崩溃

def calculate_reconstruction_and_stenosis(stenosis_lumen_mask,
                                          stenosis_wall_mask,
                                          distal_lumen_mask=None,
                                          distal_wall_mask=None,
                                          proximal_lumen_mask=None,
                                          proximal_wall_mask=None):
    """
    计算重构指数1 (基于distal), 重构指数2 (基于proximal) 和 狭窄率
    """

    # --- 狭窄层 ---
    stenosis_wall_area = calculate_wall_area1(stenosis_wall_mask)
    stenosis_lumen_area = calculate_lumen_area(stenosis_lumen_mask)
    stenosis_vessel_area = stenosis_wall_area + stenosis_lumen_area

    reconstruction_index_1, reconstruction_index_2, stenosis_rate = None, None, None
    distal_lumen_area, proximal_lumen_area = None, None

    # --- 远端参考 ---
    if distal_lumen_mask and distal_wall_mask:
        distal_wall_area = calculate_wall_area1(distal_wall_mask)
        distal_lumen_area = calculate_lumen_area(distal_lumen_mask)
        distal_vessel_area = distal_wall_area + distal_lumen_area
        if distal_vessel_area > 0:
            reconstruction_index_1 = stenosis_vessel_area / distal_vessel_area

    # --- 近端参考 ---
    if proximal_lumen_mask and proximal_wall_mask:
        proximal_wall_area = calculate_wall_area1(proximal_wall_mask)
        proximal_lumen_area = calculate_lumen_area(proximal_lumen_mask)
        proximal_vessel_area = proximal_wall_area + proximal_lumen_area
        if proximal_vessel_area > 0:
            reconstruction_index_2 = stenosis_vessel_area / proximal_vessel_area

    # --- 狭窄率 ---
    if proximal_lumen_area and distal_lumen_area and proximal_lumen_area > 0 and distal_lumen_area > 0:
        mean_lumen_area = (proximal_lumen_area + distal_lumen_area) / 2
        stenosis_rate = 1 - (stenosis_lumen_area / mean_lumen_area)
    elif distal_lumen_area and distal_lumen_area > 0:
        stenosis_rate = 1 - (stenosis_lumen_area / distal_lumen_area)
    elif proximal_lumen_area and proximal_lumen_area > 0:
        stenosis_rate = 1 - (stenosis_lumen_area / proximal_lumen_area)
    else:
        stenosis_rate = None

    return reconstruction_index_1, reconstruction_index_2, stenosis_rate


import os
import pandas as pd

def find_lumen_wall_files(folder_path):
    """找到该文件夹下的管腔和管壁文件"""
    lumen_file = None
    wall_file = None
    for file in os.listdir(folder_path):
        if file.endswith("lumen.mhd"):
            lumen_file = os.path.join(folder_path, file)
        elif file.endswith("wall.mhd"):
            wall_file = os.path.join(folder_path, file)
    return lumen_file, wall_file

def find_reference_files(folder_path):
    """找到该文件夹下的近端和远端参考文件的管腔和管壁文件"""
    distal_lumen_file = None
    distal_wall_file = None
    proximal_lumen_file = None
    proximal_wall_file = None

    for file in os.listdir(folder_path):
        if "distal" in file.lower():
            if file.endswith("lumen.mhd"):
                distal_lumen_file = os.path.join(folder_path, file)
            elif file.endswith("wall.mhd"):
                distal_wall_file = os.path.join(folder_path, file)
        elif "proximal" in file.lower():
            if file.endswith("lumen.mhd"):
                proximal_lumen_file = os.path.join(folder_path, file)
            elif file.endswith("wall.mhd"):
                proximal_wall_file = os.path.join(folder_path, file)

    return distal_lumen_file, distal_wall_file, proximal_lumen_file, proximal_wall_file

def process_case(case_folder, reference_folder):
    """
    处理单个病例，计算重构指数和狭窄率
    返回: (ri1, ri2, stenosis_rate)
    """
    # --- 狭窄层 ---
    stenosis_lumen_file, stenosis_wall_file = find_lumen_wall_files(case_folder)
    if not stenosis_lumen_file or not stenosis_wall_file:
        print(f"未找到有效的管腔和管壁文件: {case_folder}")
        return None

    # --- 参考层 ---
    distal_lumen_file, distal_wall_file, proximal_lumen_file, proximal_wall_file = find_reference_files(reference_folder)
    if not distal_lumen_file and not proximal_lumen_file:
        print(f"未找到任何有效的参考层面文件: {reference_folder}")
        return None

    # --- 计算 ---
    ri1, ri2, stenosis_rate = calculate_reconstruction_and_stenosis(
        stenosis_lumen_file, stenosis_wall_file,
        distal_lumen_file, distal_wall_file,
        proximal_lumen_file, proximal_wall_file
    )

    return ri1, ri2, stenosis_rate


def save_results_to_excel(results, output_file):
    """
    将结果保存到 Excel 文件
    :param results: 计算结果列表
    :param output_file: 输出 Excel 文件路径
    """
    # 使用 pandas 将结果转换为 DataFrame
    df = pd.DataFrame(results)
    # 保存到 Excel 文件
    df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"结果已保存到 {output_file}")

def main(stenosis_path, reference_path, output_file):
    """
    主函数，遍历狭窄层面文件夹，处理每个病例，计算所有重构指数和狭窄率
    :param stenosis_path: 狭窄层面文件夹路径
    :param reference_path: 参考层面文件夹路径
    :param output_file: 输出 Excel 文件路径
    """
    results = []

    # 遍历狭窄层面文件夹
    for case_id in os.listdir(stenosis_path):
        case_folder = os.path.join(stenosis_path, case_id)
        if not os.path.isdir(case_folder):
            continue

        # 查找参考层面对应的文件夹
        reference_folder = os.path.join(reference_path, case_id)
        if not os.path.isdir(reference_folder):
            print(f"找不到参考文件夹：{reference_folder}")
            continue

        # 处理病例，计算重构指数和狭窄率
        result = process_case(case_folder, reference_folder)
        if result:
            results.append({
                'ID': case_id,
                'Remodeling_Index_distal': result[0],
                'Remodeling_Index_proximal': result[1],
                'Stenosis_Rate': result[2]
            })

    # 保存结果到 Excel
    save_results_to_excel(results, output_file)

# 示例使用
stenosis_path = "PROJECT_ROOT/T1_MinLumen_slice"
reference_path = "PROJECT_ROOT/reference"
output_file = "PROJECT_ROOT/feature_excel/5.xlsx"

main(stenosis_path, reference_path, output_file)

结果已保存到 PROJECT_ROOT/feature_excel/5.xlsx


In [4]:
# 计算整段血管重构指数
import SimpleITK as sitk
import numpy as np
import os
import pandas as pd


# ========== 原函数：面积计算 ==========
def calculate_lumen_area(lumen_file):
    try:
        lumen_img = sitk.ReadImage(lumen_file)
        lumen_data = sitk.GetArrayFromImage(lumen_img)
        if lumen_data.ndim == 3 and lumen_data.shape[0] == 1:
            lumen_data = lumen_data[0]
        elif lumen_data.ndim == 3 and lumen_data.shape[0] > 1:
            raise ValueError("3D图像暂不支持")
        lumen_spacing = lumen_img.GetSpacing()[:2]
        voxel_area = lumen_spacing[0] * lumen_spacing[1]
        return np.sum(lumen_data > 0) * voxel_area
    except Exception as e:
        print(f"处理管腔文件失败 {lumen_file}: {str(e)}")
        return None


def calculate_wall_area1(wall_file):
    try:
        wall_img = sitk.ReadImage(wall_file)
        wall_data = sitk.GetArrayFromImage(wall_img)
        if wall_data.ndim == 3:
            if wall_data.shape[0] == 1:
                wall_data = wall_data[0]
            else:
                wall_data = wall_data[wall_data.shape[0] // 2]
        wall_binary = (wall_data > 0).astype(np.uint8)
        wall_spacing = wall_img.GetSpacing()[:2]
        wall_voxel_area = wall_spacing[0] * wall_spacing[1]
        return np.sum(wall_binary) * wall_voxel_area
    except Exception as e:
        print(f"处理管壁文件失败 {wall_file}: {str(e)}")
        return 0


# ========== 计算整段血管平均重构指数（平均面积法） ==========
def calculate_whole_vessel_remodeling_average(lumen_case_dir, wall_case_dir, reference_folder):
    vessel_areas = []
    distal_vessel_areas = []
    proximal_vessel_areas = []

    # --- 所有狭窄段切片 ---
    lumen_files = sorted([f for f in os.listdir(lumen_case_dir) if f.endswith(".mhd")])
    for fname in lumen_files:
        lumen_path = os.path.join(lumen_case_dir, fname)
        wall_fname = fname.replace("lumen", "wall")
        wall_path = os.path.join(wall_case_dir, wall_fname)
        if not os.path.exists(wall_path):
            continue
        lumen_area = calculate_lumen_area(lumen_path)
        wall_area = calculate_wall_area1(wall_path)
        if lumen_area is not None:
            vessel_areas.append(lumen_area + wall_area)

    # --- distal/proximal 参考层 ---
    for file in os.listdir(reference_folder):
        if file.endswith("lumen.mhd"):
            lumen_path = os.path.join(reference_folder, file)
            wall_path = lumen_path.replace("lumen.mhd", "wall.mhd")
            if not os.path.exists(wall_path):
                continue
            lumen_area = calculate_lumen_area(lumen_path)
            wall_area = calculate_wall_area1(wall_path)
            if "distal" in file.lower():
                distal_vessel_areas.append(lumen_area + wall_area)
            elif "proximal" in file.lower():
                proximal_vessel_areas.append(lumen_area + wall_area)

    mean_vessel_area = np.mean(vessel_areas) if len(vessel_areas) > 0 else 0
    mean_distal_area = np.mean(distal_vessel_areas) if len(distal_vessel_areas) > 0 else 0
    mean_proximal_area = np.mean(proximal_vessel_areas) if len(proximal_vessel_areas) > 0 else 0

    ri_whole_distal = mean_vessel_area / mean_distal_area if mean_distal_area > 0 else None
    ri_whole_proximal = mean_vessel_area / mean_proximal_area if mean_proximal_area > 0 else None

    return ri_whole_distal, ri_whole_proximal


# ========== 主逻辑：整段RI计算 ==========
def process_case(case_id, lumen_root, wall_root, reference_root):
    lumen_case_dir = os.path.join(lumen_root, case_id)
    wall_case_dir = os.path.join(wall_root, case_id)
    reference_case_dir = os.path.join(reference_root, case_id)

    if not os.path.isdir(lumen_case_dir) or not os.path.isdir(wall_case_dir):
        print(f"⚠️ 缺少 lumen/wall 文件夹: {case_id}")
        return None
    if not os.path.isdir(reference_case_dir):
        print(f"⚠️ 缺少 reference 文件夹: {case_id}")
        return None

    ri_whole_distal, ri_whole_proximal = calculate_whole_vessel_remodeling_average(
        lumen_case_dir, wall_case_dir, reference_case_dir
    )
    return ri_whole_distal, ri_whole_proximal


# ========== 批处理所有病例 ==========
def main(lumen_root, wall_root, reference_root, output_file):
    results = []
    case_ids = [d for d in os.listdir(lumen_root) if os.path.isdir(os.path.join(lumen_root, d))]

    for case_id in case_ids:
        ri_whole_distal, ri_whole_proximal = process_case(case_id, lumen_root, wall_root, reference_root)
        if ri_whole_distal is not None or ri_whole_proximal is not None:
            results.append({
                'ID': case_id,
                'Whole_Remodeling_Index_distal': ri_whole_distal,
                'Whole_Remodeling_Index_proximal': ri_whole_proximal
            })
            print(f"✔ {case_id}: distal={ri_whole_distal}, proximal={ri_whole_proximal}")
        else:
            print(f"⚠️ {case_id}: 未能计算")

    df = pd.DataFrame(results)
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"✅ 已保存结果至 {output_file}")


# ========== 运行入口 ==========
if __name__ == "__main__":
    lumen_root = "PROJECT_ROOT/lumen"
    wall_root = "PROJECT_ROOT/wall"
    reference_root = "PROJECT_ROOT/reference"
    output_file = "PROJECT_ROOT/feature_excel/whole_remodeling_index.xlsx"

    main(lumen_root, wall_root, reference_root, output_file)



✔ wangzhaosong_0006229548: distal=0.42752406307597796, proximal=0.7881899871630296
✔ peixufeng_0010408826: distal=1.311624649859944, proximal=0.983718487394958
✔ wangzibing_0010678758: distal=0.8570846905537458, proximal=0.7571942446043165
✔ liuyouzhen_0008865819: distal=1.0447530864197532, proximal=0.8273393854748604
✔ wangsongcai_0010310763: distal=1.1586101398601396, proximal=0.8946071814254859
✔ yuanyongxia_0007899705: distal=0.6300795117800663, proximal=0.8703140830800404
✔ nixiangfei_0012199726: distal=0.7309448542325255, proximal=0.5562683774391874
✔ liushiping_0010366367: distal=0.9508559543491013, proximal=1.2441211359988835
✔ liuyehua_0010381650: distal=0.8688561215370867, proximal=0.8173602353930223
✔ chenshuangmei_0008974237: distal=0.4659624413145541, proximal=0.5173761946133798
✔ xufuyong_800610: distal=1.133601217366494, proximal=1.0009027165470044
✔ heruohuai_0010814687: distal=0.9981322634200332, proximal=1.1573754712440842
✔ jiangqijun_0009742087: distal=0.88642911296

#### 管腔直径

In [ ]:
import os
import re
import numpy as np
import cv2
import SimpleITK as sitk
import pandas as pd
from scipy.ndimage import binary_erosion

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)[_.]', filename)
    return int(match.group(1))

def calculate_slice_diameters(original_path, mask_path,case_id):
    """计算单个切片的直径"""
    try:
        # 读取mask
        mask_image = sitk.ReadImage(mask_path)
        mask = sitk.GetArrayFromImage(mask_image)
        mask = np.squeeze(mask)
        
        img_spacing = mask_image.GetSpacing()[:2]
        spacing_x = img_spacing[0]
        spacing_y = img_spacing[1]
        
        # 检查有效性
        if mask.ndim != 2 or np.sum(mask) == 0:
            return 0.0, 0.0, 0.0
        
        # 中心点计算逻辑（保持原有算法）
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"🚫{case_id}存在管腔直径为0的情况")
            return 0.0, 0.0, 0.0
        
        main_contour = max(contours, key=cv2.contourArea)
        M = cv2.moments(main_contour)
        if M["m00"] == 0:
            return 0.0, 0.0, 0.0
        
        # 计算中心点
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        
        # 计算各方向半径
        max_radius = np.zeros(360)
        contour_points = main_contour.squeeze(1)  # 假设轮廓形状为(N,1,2)
        
        for (x, y) in contour_points:
            dx = (x - cx)*spacing_x
            dy = (y - cy)*spacing_y
            theta = int(np.round(np.degrees(np.arctan2(dy, dx)) % 360))
            radius = np.hypot(dx, dy)
            if radius > max_radius[theta]:
                max_radius[theta] = radius
        
        # 计算有效直径
        diameters = []
        for theta in range(180):
            opp_theta = (theta + 180) % 360
            if max_radius[theta] > 0 or max_radius[opp_theta] > 0:
                diameters.append(max_radius[theta] + max_radius[opp_theta])
        
        return max(diameters), min(diameters), np.mean(diameters)
    
    except Exception as e:
        print(f"处理切片失败 {mask_path}: {str(e)}")
        return 0.0, 0.0, 0.0

def process_case(case_id, original_dir, mask_dir):
    """处理单个病例的所有切片"""
    case_diameters = []
    
    # 获取病例路径
    original_case_path = os.path.join(original_dir, case_id)
    mask_case_path = os.path.join(mask_dir, case_id)
    
    # 修改部分：按切片编号排序 ===========================================
    # 获取所有mask文件并排序
    mask_files = []
    for f in os.listdir(mask_case_path):
        if f.endswith('.mhd'):
            slice_idx = find_slice_number(f)
            mask_files.append((slice_idx, f))
    
    # 按切片编号升序排序
    mask_files.sort(key=lambda x: x[0])
    
    # 遍历排序后的切片 ==================================================
    for slice_idx, mask_file in mask_files:
        # 匹配原始图像文件
        original_files = [f for f in os.listdir(original_case_path) 
                         if find_slice_number(f) == slice_idx and f.endswith('.mhd')]
        
        if not original_files:
            print(f"找不到原始图像：{case_id} 切片{slice_idx}")
            continue
            
        original_path = os.path.join(original_case_path, original_files[0])
        mask_path = os.path.join(mask_case_path, mask_file)
        
        # 计算直径
        max_d, min_d, avg_d = calculate_slice_diameters(original_path, mask_path,case_id)
        
        # 记录有效结果
        # if max_d > 0:
        case_diameters.append((slice_idx, max_d, min_d, avg_d))  # 记录切片编号
        print(f"{case_id} 切片{slice_idx:03d}: "
                f"最大直径={max_d:.2f}mm | "
                f"最小直径={min_d:.2f}mm | "
                f"平均直径={avg_d:.2f}mm")
    
    # 统计病例结果
    if not case_diameters:
        print(f"⚠️ {case_id} 无有效切片")
        return None
    
    # 按切片编号排序结果 ================================================
    case_diameters.sort(key=lambda x: x[0])  # 按第一个元素（slice_idx）排序
    
    # 提取直径数据（已排序）
    max_values = [d[1] for d in case_diameters]
    min_values = [d[2] for d in case_diameters]
    avg_values = [d[3] for d in case_diameters]
    
    return {
        'ID': case_id,
        'Slices': len(case_diameters),
        'lumen_Diameter_max_Total(mm)': round(np.max(max_values), 2),
        'lumen_Diameter_min_Total(mm)': round(np.min(min_values), 2),
        'lumen_Diameter_mean_Total(mm)': round(np.mean(avg_values), 2)
    }

# 路径配置
original_image_dir = "PROJECT_ROOT/T1_slice"  # 原始图像路径
lumen_mask_dir = "PROJECT_ROOT/lumen"         # 管腔mask路径
output_excel = "PROJECT_ROOT/feature_excel/6.xlsx" # 输出文件路径

if __name__ == "__main__":
    results = []
    
    # 遍历所有病例
    for case_id in os.listdir(lumen_mask_dir):
        # 跳过非目录文件
        if not os.path.isdir(os.path.join(lumen_mask_dir, case_id)):
            continue
            
        # 处理病例
        case_result = process_case(case_id, original_image_dir, lumen_mask_dir)
        if case_result:
            results.append(case_result)
            print(f"✅ {case_id} 处理完成")
            print('-'*60)
        else:
            print(f"⚠️{case_id}没有获取到直径信息！")
    # 生成报告
    if results:
        df = pd.DataFrame(results)
        df = df[[
            'ID', 'Slices',
            'lumen_Diameter_max_Total(mm)', 'lumen_Diameter_min_Total(mm)',
            'lumen_Diameter_mean_Total(mm)'
        ]]
        df.to_excel(output_excel, index=False)
        print(f"\n成功处理 {len(df)} 个病例")
        print(f"结果保存至：{output_excel}")
    else:
        print("\n没有找到有效病例数据")

#### 管壁直径

In [ ]:
import numpy as np
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
import os
import pandas as pd
import re

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)[_.]', filename)
    return int(match.group(1))

def calculate_wall_diameters(original_file_path, wall_file_path, id_dir,slice_idx):
    image = sitk.ReadImage(wall_file_path)
    mask = sitk.GetArrayFromImage(image)
    img_spacing = image.GetSpacing()[:2]
    spacing_x, spacing_y = img_spacing

    if mask.ndim == 3:
        slice_areas = [np.count_nonzero(slice) for slice in mask]
        chosen_slice = np.argmax(slice_areas)
        mask = mask[chosen_slice, :, :]

    mask = mask.astype(np.uint8)
    _, mask_bin = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY)
    height, width = mask_bin.shape

    contours_wall, hierarchy = cv2.findContours(mask_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if not contours_wall:
        print(f"{id_dir}的切片{slice_idx}🚫❌未检测到管壁轮廓")
        return 0,0,0
        #raise ValueError(f"{id_dir}的切片{slice_idx}🚫❌未检测到管壁轮廓")
        #return 0,0,0

    outer_contours = [c for i, c in enumerate(contours_wall) if hierarchy[0][i][3] == -1]
    inner_contours = [c for i, c in enumerate(contours_wall) if hierarchy[0][i][3] != -1]

    if not outer_contours:
        raise ValueError(f"{id_dir}的切片{slice_idx}🚫❌未检测到管壁外边界")
    largest_outer = max(outer_contours, key=cv2.contourArea)

    # 智能中心点调整 ====================================================
    initial_center = (width//2, height//2)  # 初始中心点为图像中心
    center_x, center_y = initial_center
    
    # 存在内轮廓时的处理
    if inner_contours:
        #print(f"{id_dir}✔️直接使用内轮廓计算")
        # 只处理最大内轮廓
        largest_inner = max(inner_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_inner, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_inner)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_inner, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_inner)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_inner, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以内轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算(内轮廓)")
    elif outer_contours and not inner_contours:
        print(f"{id_dir}✔️未检测到内轮廓,直接使用外轮廓计算")
         # 只处理最大内轮廓
        # largest_outer = max(outer_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_outer, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_outer)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_outer, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_outer)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_outer, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以外轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算")
        
    else:
        print(f"{id_dir}的切片{slice_idx}🚫❌内轮廓和外轮廓都未检测到！")

    # 投影法计算各方向最大半径
    max_radius = np.zeros(360)
    for theta in range(360):
        theta_rad = np.radians(theta)
        direction = np.array([np.cos(theta_rad), np.sin(theta_rad)])
        max_proj = -np.inf
        
        for point in largest_outer:
            x, y = point[0]
            dx = (x - center_x) * spacing_x
            dy = (y - center_y) * spacing_y
            proj = dx * direction[0] + dy * direction[1]
            if proj > max_proj:
                max_proj = proj
        
        if max_proj > 0:
            max_radius[theta] = max_proj

    diameters = []
    for theta in range(180):
        opposite_theta = theta + 180
        if max_radius[theta] > 0 and max_radius[opposite_theta] > 0:
            diameters.append(max_radius[theta] + max_radius[opposite_theta])
    
    if not diameters:
        raise ValueError(f"{id_dir}🚫❌直径计算出错")
    
    return np.max(diameters), np.min(diameters), np.mean(diameters)

def process_case(case_id, original_path,wall_path):
    case_wall_diameters = []
    #获取病例路径
    wall_files_path = os.path.join(wall_path,id_dir)
    original_files_path = os.path.join(original_path, id_dir)
    #获取所有mask文件并排序
    mask_files = []
    for f in os.listdir(wall_files_path):
        if f.endswith('mhd'):
            slice_idx = find_slice_number(f)
            mask_files.append((slice_idx,f))
    #按切片编号升序排序
    mask_files.sort(key=lambda x: x[0])
    
    for slice_idx, mask_file in mask_files:
        original_files = [f for f in os.listdir(original_files_path) if f.endswith('.mhd') and find_slice_number(f) == slice_idx]
        if not original_files:
            print(f"找不到原始图像：{case_id} 切片{slice_idx}")
            continue
        original_file_path = os.path.join(original_files_path,original_files[0])
        mask_path = os.path.join(wall_files_path,mask_file)
         
        if not (os.path.exists(original_file_path) and os.path.exists(mask_path)):
            print(f"跳过 {original_file_path}: 缺少对应的original文件")

        #计算直径
        wall_max, wall_min, wall_avg = calculate_wall_diameters(original_file_path, mask_path, id_dir,slice_idx)
        
        #记录有效结果
        case_wall_diameters.append((slice_idx,wall_max,wall_min,wall_avg))
        print(f"{case_id} 切片{slice_idx:03d}: "
                f"最大管壁直径={wall_max:.2f}mm | "
                f"最小管壁直径={wall_min:.2f}mm | "
                f"平均管壁直径={wall_avg:.2f}mm")
    
    if not case_wall_diameters:
        print(f"⚠️ {case_id} 无有效管壁切片")
        return None
        
    #按切片编号排序结果===========================
    case_wall_diameters.sort(key=lambda x: x[0]) #按第一个元素(slice_idx)排序
    
    #提取直径数据
    max_values = [d[1] for d in case_wall_diameters]  #d[1]最大管壁直径
    min_values = [d[2] for d in case_wall_diameters]  #d[2]最小管壁直径
    avg_values = [d[3] for d in case_wall_diameters]  #d[3]平均管壁直径

    return{
        'ID':case_id,
        'Slices':len(case_wall_diameters),
        'wall_Diameter_max_Total(mm)': round(np.max(max_values),2),
        'wall_Diameter_min_Total(mm)': round(np.min(min_values),2),
        'wall_Diameter_mean_Total(mm)': round(np.mean(avg_values),2)
    }

#路径配置
original_path = "PROJECT_ROOT/T1_slice"
wall_path = "PROJECT_ROOT/wall"
output_path = "PROJECT_ROOT/feature_excel"
output_excel = os.path.join(output_path, "7.xlsx")

results = []
for id_dir in os.listdir(wall_path):
    print(f"⚪正在处理{id_dir}")
    
    if not os.path.isdir(os.path.join(wall_path,id_dir)):
        continue
    
    case_result = process_case(id_dir,original_path,wall_path)
    if case_result:
        results.append(case_result)
        print(f"✅ {id_dir} 处理完成")
        print('-'*60)
    else:
        print(f"⚠️{id_dir}没有获取到管壁直径信息！")
    
if results:
    df = pd.DataFrame(results)
    df = df[[
            'ID', 'Slices',
            'wall_Diameter_max_Total(mm)', 'wall_Diameter_min_Total(mm)',
            'wall_Diameter_mean_Total(mm)'
        ]]
    df.to_excel(output_excel, index=False)
    print(f"结果已保存至 {output_excel}")
else:
    print("无有效数据可保存")

#### 管壁厚度

In [ ]:
import numpy as np
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

def visualize_masks(image_path, mask_path, center_x, center_y):
    """可视化图像和mask轮廓，并在图像上显示中心点"""
    image = sitk.GetArrayFromImage(sitk.ReadImage(image_path))
    image = np.squeeze(image)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    mask = np.squeeze(mask)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='r', linewidths=0.5)
    # 在图像上绘制中心点，'ro'表示红色的点
    plt.plot(center_x, center_y, 'go',markersize=1)  # 红色点ro
    plt.title('Image with Mask and Center Point')
    plt.show()
    plt.close()


def find_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)[_.]', filename)
    return int(match.group(1))

# 改进的半径计算 ====================================================
def process_contour(contours, img_spacing,center, is_outer=True):
    radius_array = np.zeros(360) if is_outer else np.full(360, np.inf)
    all_points = []
    
    spacing_x = img_spacing[0]
    spacing_y = img_spacing[1]
    
    for contour in contours:
        points = contour.squeeze()
        if points.ndim != 2 or points.shape[1] != 2:
            continue
        all_points.extend(points.tolist())
    
    if not all_points:
        return radius_array
    
    # 转换为极坐标
    points_array = np.array(all_points)
    dx = (points_array[:, 0] - center[0])*spacing_x
    dy = (points_array[:, 1] - center[1])*spacing_y
    radii = np.hypot(dx, dy)
    thetas = np.degrees(np.arctan2(dy, dx)) % 360

    # 改进的角度分配策略：分配到最近的三个角度
    for theta, r in zip(thetas, radii):
        base_angle = int(np.round(theta))
        for offset in [-1, 0, 1]:  # 覆盖三个相邻角度
            angle = (base_angle + offset) % 360
            if is_outer:
                if r > radius_array[angle]:
                    radius_array[angle] = r
            else:
                if r < radius_array[angle]:
                    radius_array[angle] = r
    return radius_array

def calculate_wall_thickness(original_file_path,mask_path,id_dir,slice_idx):
    # original_file_path是以mhd结尾的文件路径了
    
    # 读取 .mhd 文件
    image = sitk.ReadImage(mask_path)
    mask = sitk.GetArrayFromImage(image)
    img_spacing = image.GetSpacing()[:2]
    #spacing_x, spacing_y = img_spacing
    
    # 处理三维数据
    if mask.ndim == 3:
        mask = mask[0, :, :]  # 如果是三维数据，只取第一个切片
        
    mask = mask.astype(np.uint8)  # 将数据类型转换为无符号8位整型
    #进行二值化
    _, mask_bin = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY)
    
    height, width = mask_bin.shape  # 获取图像的高度和宽度
    
    # 检测轮廓
    contours_wall, hierarchy = cv2.findContours(mask_bin, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if not contours_wall:
        print(f"{id_dir}的切片{slice_idx}🚫❌未检测到管壁轮廓")
        return 0,0,0
    
    # 分离外轮廓和内轮廓
    outer_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] == -1]  # 外轮廓列表
    inner_contours = [c for i,c in enumerate(contours_wall) if hierarchy[0][i][3] != -1]  # 内轮廓列表
    
    # 检查外轮廓存在性
    if not outer_contours:
        #没有外轮廓，肯定没有内轮廓，应该是没有标签，为空
        print(f"❌❌{id_dir}未检测到任何外轮廓")
        return 0,0,0
    else:
        largest_outer = max(outer_contours, key=cv2.contourArea)

    # 智能中心点调整 ====================================================
    initial_center = (width//2, height//2)  # 初始中心点为图像中心
    center_x, center_y = initial_center
    
    # 存在内轮廓时的处理
    if inner_contours:
        #print(f"{id_dir}✔️直接使用内轮廓计算")
        # 只处理最大内轮廓
        largest_inner = max(inner_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_inner, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_inner)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_inner, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_inner)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_inner, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以内轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算(内轮廓)")
    elif outer_contours and not inner_contours:
        print(f"{id_dir}✔️未检测到内轮廓,直接使用外轮廓计算")
         # 只处理最大内轮廓
        # largest_outer = max(outer_contours, key=cv2.contourArea)  # 找到面积最大的内轮廓
        center_status = cv2.pointPolygonTest(largest_outer, initial_center, False)  # 检查初始中心点是否在内轮廓内
        
        if center_status <= 0:  # 如果初始中心点在内轮廓外
            # 三级中心调整策略
            # 1. 质心法
            M = cv2.moments(largest_outer)  # 计算轮廓的矩
            if M["m00"] > 0:  # 防止除以零
                cx = int(M["m10"]/M["m00"])  # 质心x坐标
                cy = int(M["m01"]/M["m00"])  # 质心y坐标
                if cv2.pointPolygonTest(largest_outer, (cx, cy), False) >= 0:  # 检查质心是否在内轮廓内
                    center_x, center_y = cx, cy
                    #print("使用质心作为新中心")
            
            # 2. 包围盒中心法（如果质心法失败）
            if (center_x, center_y) == initial_center:  # 如果质心法没有更新中心点
                x,y,w,h = cv2.boundingRect(largest_outer)  # 计算轮廓的包围盒
                new_center = (x + w//2, y + h//2)  # 计算包围盒中心
                if cv2.pointPolygonTest(largest_outer, new_center, False) >= 0:  # 检查包围盒中心是否在内轮廓内
                    center_x, center_y = new_center
                    #print("使用包围盒中心作为新中心")
            
            # 3. 距离变换法（最后保障）
            if (center_x, center_y) == initial_center:  # 如果前两种方法都没有更新中心点
                dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 3)  # 计算距离变换
                _, _, _, max_loc = cv2.minMaxLoc(dist_map)  # 找到距离变换的最大值位置
                center_x, center_y = max_loc
                #print("使用距离变换中心作为新中心")
            print(f"{id_dir}✔️以外轮廓为基准修改了中心")
        else:
            print(f"{id_dir}✔️使用原中心计算")    
    else:
        print(f"{id_dir}的切片{slice_idx}🚫❌内轮廓和外轮廓都未检测到！")
        
    # 可视化 ==========================================================
    # if id_dir == '0000864133' and slice_idx == 14:
    #     #original_path需要的是mhd文件
    #     visualize_masks(original_file_path, mask_path, center_x, center_y)
    
    # 计算半径 ========================================================
    #投影法计算个方向最大管壁厚度
    # 处理外轮廓
    #前面已经判断了，如果没有外轮廓，到不了这一步的，这里不用判断了
    max_radius = process_contour(outer_contours,img_spacing, (center_x, center_y), is_outer=True)
    
    # 处理内轮廓（智能插值）
    if inner_contours:
        # 初步处理内轮廓
        min_radius = process_contour(inner_contours, img_spacing,(center_x, center_y), is_outer=False)
        
        # 创建有效数据点集合
        valid_mask = min_radius != np.inf
        valid_angles = np.where(valid_mask)[0]
        valid_values = min_radius[valid_mask]
        
        # 智能插值策略（仅在外轮廓有效区域）
        outer_mask = max_radius > 1e-6
        if len(valid_angles) > 10:
            # 扩展角度处理周期边界
            extended_angles = np.concatenate([valid_angles-360, valid_angles, valid_angles+360])
            extended_values = np.concatenate([valid_values, valid_values, valid_values])
            
            # 创建插值函数
            f = interp1d(extended_angles, extended_values, 
                        kind='linear', bounds_error=False)
            
            # 仅在外轮廓有效区域插值
            need_interp = np.where(outer_mask & ~valid_mask)[0]
            if len(need_interp) > 0:
                interpolated = f(need_interp)
                interpolated = np.clip(interpolated, 0, max_radius[need_interp]*0.95)
                min_radius[need_interp] = interpolated
    else:
        #内轮廓的半径用min_radius表示，外轮廓半径用max_radius表示
        #如果没有内轮廓，则管壁厚度就是外轮廓的半径
        min_radius = np.zeros_like(max_radius)
        print(f"{id_dir}没有内轮廓，直接用外轮廓半径作为管壁厚度")

    # 改进的壁厚计算 ====================================================
    wall_thickness = []
    for theta in range(360):
        outer_r = max_radius[theta]
        if outer_r <= 1e-6:
            continue
            
        if inner_contours:
            inner_r = min_radius[theta]
            
            # 异常值处理策略，本来有等于条件
            if inner_r > outer_r:
                print(f"❌❌❌❌❌❌❌{id_dir}的切片{slice_idx}内轮廓半径大于外轮廓半径！不可能呀，哥们，检查一下图像吧")
                # 三级修复策略
                # 1. 寻找相邻有效值
                neighbor_values = []
                for offset in [-2, -1, 1, 2]:
                    neighbor_theta = (theta + offset) % 360
                    if min_radius[neighbor_theta] < outer_r:
                        neighbor_values.append(min_radius[neighbor_theta])
                
                # 2. 使用统计值
                if neighbor_values:
                    inner_r = np.mean(neighbor_values)
                else:
                    # 3. 安全回退策略
                    inner_r = outer_r * 0.8
                
                # 最终保障
                inner_r = min(inner_r, outer_r*0.95)
            
            thickness = outer_r - inner_r
            wall_thickness.append(thickness)
        else:
            wall_thickness.append(outer_r)
    
    #这里考虑到了没有标签的情况，输出000        
    if wall_thickness:
        thickness_max_d = max(wall_thickness)  # 计算最大直径
        thickness_min_d = min(wall_thickness)  # 计算最小直径
        thickness_avg_d = np.mean(wall_thickness)  #计算平均直径
        return thickness_max_d,thickness_min_d,thickness_avg_d
    else:
        print(f"{id_dir}❌❌没有分割啥东西出来哦")
        return 0.0,0.0,0.0


def process_case(id_dir,original_path,case_path):
    case_thincknesses = []
    #original_files_path加上ID
    original_files_path = os.path.join(original_path,id_dir)
    wall_files_path = case_path
    #获取所有的mask文件并排序
    mask_files = []
    for f in os.listdir(wall_files_path):
        if f.endswith('mhd'):
            slice_idx = find_slice_number(f)
            mask_files.append((slice_idx,f))
    #按切片编号升序排序
    mask_files.sort(key=lambda x: x[0])
    
    for slice_idx,mask_file in mask_files:
        #获取编号相同的原始图像
        original_files = [f for f in os.listdir(original_files_path) if f.endswith('.mhd') and find_slice_number(f) == slice_idx]
        if not original_files:
            print(f"⚠️找不到原始图像：{id_dir} 切片{slice_idx}")
            continue
        original_file_path = os.path.join(original_files_path,original_files[0])
        mask_path = os.path.join(wall_files_path,mask_file)

        if not (os.path.exists(original_file_path) and os.path.exists(mask_path)):
            print(f"⚠️跳过 {original_file_path}: 缺少对应的original文件")

        #计算管壁厚度(要处理没有管壁mask的情况哈)
        max_t, min_t, avg_t = calculate_wall_thickness(original_file_path, mask_path, id_dir,slice_idx)
        #计算偏心指数
        if max_t > 0:
            eccentricity_index_total = (max_t-min_t)/max_t
        else:
            eccentricity_index_total =0
        
        #记录有效结果
        case_thincknesses.append((slice_idx,max_t, min_t, avg_t,eccentricity_index_total))
        print(f"{id_dir} 切片{slice_idx:03d}: "
                f"最大管壁厚度={max_t:.4f}mm | "
                f"最小管壁厚度={min_t:.4f}mm | "
                f"平均管壁厚度={avg_t:.4f}mm | "
                f"偏心指数={eccentricity_index_total:.4f}")
    
    if not case_thincknesses:
        print(f"⚠️ {id_dir} 无有效管壁切片")
        return None
    
    #按切片编号排序结果====================================
    case_thincknesses.sort(key=lambda x: x[0]) #按第一个元素(slice_idx)排序
    
    #提取管壁厚度数据
    max_values = [d[1] for d in case_thincknesses]  #d[1]最大管壁厚度
    min_values = [d[2] for d in case_thincknesses]  #d[2]最小管壁厚度
    avg_values = [d[3] for d in case_thincknesses]  #d[3]平均管壁厚度
    eccentricity_index_total = [d[4] for d in case_thincknesses] #d[4]偏心指数

    return{
        'ID':case_id,
        'Slices':len(case_thincknesses),
        'wall_thickness_max_total(mm)': round(np.max(max_values),2),
        'wall_thickness_min_total(mm)': round(np.min(min_values),2),
        'wall_thickness_avg_total(mm)': round(np.mean(avg_values),2),
        'eccentricity_index_total': round(np.max(eccentricity_index_total),2)
    }
    

#设置路径
original_path = "PROJECT_ROOT/T1_slice"
wall_path = "PROJECT_ROOT/wall"
output_path = "PROJECT_ROOT/feature_excel"
output_excel = os.path.join(output_path, "8.xlsx")


# 主处理流程 ====================================================
results = []
for case_id in os.listdir(wall_path):
    case_dir = os.path.join(wall_path, case_id)
    if not os.path.isdir(case_dir):
        continue
    
    # print(f"\n{'='*40}")
    print(f"⚪开始处理病例: {case_id}")
    
    #case_dir已经包含ID文件夹了
    #original_path不包含ID
    case_result = process_case(case_id,original_path,case_dir)
    
    if case_result:
        results.append(case_result)
        print(f"✅ {case_id} 处理完成")
    else:
        print(f"❌ {case_id} 没有获取到管壁厚度信息！")
    
    print(f"{'='*40}\n")

# 结果保存 ======================================================
if results:
    df = pd.DataFrame(results)
    df = df[['ID', 'Slices', 
             'wall_thickness_max_total(mm)', 
             'wall_thickness_min_total(mm)',
             'wall_thickness_avg_total(mm)',
             'eccentricity_index_total']]
    
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='Wall Thickness')
        
    print(f"结果已保存至: {output_excel}")
else:
    print("无有效数据可保存")

#### 面积特征

In [ ]:
import os
import re
import cv2
import numpy as np
import SimpleITK as sitk
import pandas as pd

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)[_.]', filename)
    return int(match.group(1)) if match else None

def calculate_area(file):
    """通用函数计算管壁面积和检测内轮廓"""
    try:
        file_img = sitk.ReadImage(file)
        data = sitk.GetArrayFromImage(file_img)
        
        if data.ndim == 3 and data.shape[0] == 1:
            data = data[0]
        elif data.ndim == 3 and data.shape[0] > 1:
            raise ValueError("3D图像暂不支持")
        
        wall_binary = ((data > 0) * 255).astype(np.uint8)
        
        wall_spacing = file_img.GetSpacing()[:2]
        wall_voxel_area = wall_spacing[0] * wall_spacing[1]
        wall_area = np.sum(wall_binary > 0) * wall_voxel_area
        
        return wall_area
    except Exception as e:
        print(f"处理管壁文件失败 {file}: {str(e)}")
        return None, None

def process_single_slice(lumen_path, wall_path, case_id,slice_idx):
    """处理单个切片，计算所有指标"""
    try:
        lumen_area = calculate_area(lumen_path)
        if lumen_area == 0:
            print(f"🚫❌{case_id}的{slice_idx}切片管腔面积为0")
        
        # 处理管壁mask
        wall_area = calculate_area(wall_path)
        if wall_area == 0:
            print(f"🚫❌{case_id}的{slice_idx}切片管壁面积为0")
            
        # 计算血管面积和NWI
        vessel_area = lumen_area + wall_area
        if vessel_area:
            nwi = wall_area / vessel_area
        else:
            print(f"🚫❌{case_id}的{slice_idx}切片血管面积为0，无法计算NWI")
            nwi = 0
        
        return {
            'lumen_area(mm²)_Total': lumen_area,
            'wall_area(mm²)_Total': wall_area,
            'vessel_area(mm²)_Total': vessel_area,
            'nwi_Total': nwi,
            # 'lumen_file': os.path.basename(lumen_path),
            # 'wall_file': os.path.basename(wall_path)
        }
    except Exception as e:
        print(f"处理{case_id}的{slice_idx}切片失败 : {str(e)}")
        return None

def process_id_folder(lumen_case_path, wall_case_path, case_id):
    """处理单个病例文件夹，计算所有切片的指标"""
    slice_features = []
    lumen_files = [f for f in os.listdir(lumen_case_path) if f.endswith(".mhd")]

    for lumen_file in lumen_files:
        slice_idx = find_slice_number(lumen_file)
        if slice_idx is None:
            continue

        lumen_path = os.path.join(lumen_case_path, lumen_file)
        
        # 查找对应的管壁文件
        wall_file = None
        for f in os.listdir(wall_case_path):
            if f.endswith(".mhd") and find_slice_number(f) == slice_idx:
                wall_file = f
                break
                
        if not wall_file:
            print(f"{slice_idx}🚫❌没找到对应的管壁文件")
            continue
            
        wall_path = os.path.join(wall_case_path, wall_file)
        features = process_single_slice(lumen_path, wall_path, case_id,slice_idx)
        
        if features:
            features['slice_idx'] = slice_idx
            slice_features.append(features)

    if not slice_features:
        print(f"没有有效切片: {case_id}")
        return None
        
    # 计算统计量
    stats = {
        'Case ID': case_id,
        'Total Slices': len(slice_features)
    }
    
    # 为每个指标计算统计量
    metrics = ['lumen_area(mm²)_Total', 'wall_area(mm²)_Total', 'vessel_area(mm²)_Total', 'nwi_Total']
    for metric in metrics:
        values = [s[metric] for s in slice_features]
        stats[f'Max_{metric}'] = round(max(values), 4)
        stats[f'Min_{metric}'] = round(min(values), 4)
        stats[f'Mean_{metric}'] = round(np.mean(values), 4)
    
    return stats

# 设置路径
lumen_mask_path = "PROJECT_ROOT/lumen"
wall_mask_path = "PROJECT_ROOT/wall"
excel_path = "PROJECT_ROOT/feature_excel"

all_stats = []

for case_id in os.listdir(lumen_mask_path):
    lumen_case_path = os.path.join(lumen_mask_path, case_id)
    wall_case_path = os.path.join(wall_mask_path, case_id)

    if not os.path.isdir(lumen_case_path) or not os.path.isdir(wall_case_path):
        print(f"{case_id}🚫❌管腔或管壁路径不存在！")
        continue

    stats = process_id_folder(lumen_case_path, wall_case_path, case_id)
    if stats:
        all_stats.append(stats)
    print(f"{case_id}✅处理完成")    
    print('-'*60)
if all_stats:
    df = pd.DataFrame(all_stats)
    output_excel = os.path.join(excel_path, "9.xlsx")
    df.to_excel(output_excel, index=False)
    print(f"\n统计结果已保存到: {output_excel}")
    print(f"总病例数: {len(df)}")
else:
    print("\n警告：未找到任何有效病例数据")

print("\n处理完成!")

#### 体积

In [ ]:
import os
import re
import cv2
import numpy as np
import SimpleITK as sitk
import pandas as pd

def find_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)[_.]', filename)
    return int(match.group(1)) if match else None

def calculate_volume(file_path):
    """计算单个切片的体积（面积×层厚）"""
    try:
        img = sitk.ReadImage(file_path)
        data = sitk.GetArrayFromImage(img)
        
        # 处理维度
        if data.ndim == 3 and data.shape[0] == 1:
            data = data[0]
        elif data.ndim == 3 and data.shape[0] > 1:
            raise ValueError("3D图像暂不支持")
        
        # 计算面积
        spacing = img.GetSpacing()
        xy_area = spacing[0] * spacing[1]
        slice_area = np.sum(data > 0) * xy_area
        
        # 获取层厚（假设第三个维度为层厚）
        z_thickness = 0.5
        
        return slice_area * z_thickness
    except Exception as e:
        print(f"计算体积失败 {file_path}: {str(e)}")
        return None

def process_case(lumen_case_path, wall_case_path, case_id):
    """处理单个病例的所有切片"""
    total_lumen = 0.0
    total_wall = 0.0
    valid_slices = 0
    
    for lumen_file in os.listdir(lumen_case_path):
        #这里应该是剔除.raw文件
        if not lumen_file.endswith(".mhd"):
            #print(f"⚠️{case_id}路径下没有mhd文件")
            continue
            
        # 获取切片编号
        slice_idx = find_slice_number(lumen_file)
        if slice_idx is None:
            print(f"⚠️{case_id}的{lumen_file}没有编号")
            continue
            
        # 构建完整路径
        lumen_path = os.path.join(lumen_case_path, lumen_file)
        
        # 查找对应的管壁文件
        wall_file = None
        for f in os.listdir(wall_case_path):
            if f.endswith(".mhd") and find_slice_number(f) == slice_idx:
                wall_file = f
                break
        if not wall_file:
            print(f"{case_id}的{slice_idx}没找到对应的管壁文件")
            continue
            
        wall_path = os.path.join(wall_case_path, wall_file)
        
        # 计算体积
        lumen_vol = calculate_volume(lumen_path)
        wall_vol = calculate_volume(wall_path)
        
        if lumen_vol==0:
            print(f"🚫❌{case_id}的{slice_idx}切片的管腔体积为0")
        if wall_vol==0:
            print(f"🚫❌{case_id}的{slice_idx}切片的管壁体积为0")
            #continue
            
        # 累加体积
        total_lumen += lumen_vol
        total_wall += wall_vol
        valid_slices += 1

    if valid_slices == 0:
        print(f"⚠️ 警告：{case_id} 无有效切片")
        return None

    # 斑块长度 = 层数 × 层厚
    z_thickness = 0.5
    plaque_length = valid_slices * z_thickness
        
    return {
        'ID': case_id,
        'Slices': valid_slices,
        'Lumen_Volume(mm³)': round(total_lumen, 2),
        'Wall_Volume(mm³)': round(total_wall, 2),
        'Vessel_Volume(mm³)': round(total_lumen + total_wall, 2),
        'Plaque_Length(mm)': round(plaque_length, 2)
    }

# 路径设置
lumen_dir = "PROJECT_ROOT/lumen"
wall_dir = "PROJECT_ROOT/wall"
output_excel = "PROJECT_ROOT/feature_excel/10.xlsx"

results = []

# 遍历所有病例
for case_id in os.listdir(lumen_dir):
    lumen_path = os.path.join(lumen_dir, case_id)
    wall_path = os.path.join(wall_dir, case_id)
    
    if not os.path.isdir(lumen_path) or not os.path.isdir(wall_path):
        print(f"⚠️{case_id}管腔或管壁路径不存在")
        continue
        
    case_result = process_case(lumen_path, wall_path, case_id)
    if case_result:
        results.append(case_result)
        print(f"✅ {case_id} 处理完成")
        print('-'*60)
    else:
        print(f"⚠️{case_id}没有获取到体积信息！")

# 生成报告
if results:
    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False)
    print(f"\n成功处理 {len(df)} 个病例")
    print(f"结果已保存至：{output_excel}")
else:
    print("\n没有找到有效病例数据")

#### 求强化比例

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import pandas as pd
import re
import matplotlib.pyplot as plt

def load_slice(file_path):
    """加载.mhd文件并返回2D numpy数组"""
    image = sitk.ReadImage(file_path)
    array = sitk.GetArrayFromImage(image)
    array = np.squeeze(array)
    if array.ndim != 2:
        raise ValueError(f"文件 {file_path} 不是2D切片")
    return array

def extract_slice_number(filename):
    """从文件名中提取切片编号"""
    match = re.search(r'slice_(\d+)', filename)
    if match:
        return int(match.group(1))
    return None

def visualize_masks(image_path, mask_path):
    """可视化图像和mask轮廓"""
    image = sitk.GetArrayFromImage(sitk.ReadImage(image_path))
    image = np.squeeze(image)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
    mask = np.squeeze(mask)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(121)
    plt.imshow(image, cmap='gray')
    plt.contour(mask, colors='r', linewidths=0.5)
    plt.title('Image with Mask')
    
    # plt.subplot(122)
    # plt.imshow(mask, cmap='jet', alpha=0.5)
    # plt.title('Mask Overlay')
    plt.show()
    plt.close()

def process_patient(t1ce_image_dir, t1ce_mask_dir, t1_image_dir, t1_mask_dir, output_root, patient_id):
    output_dir = os.path.join(output_root, patient_id)
    os.makedirs(output_dir, exist_ok=True)
    
    max_t1ce_value = -np.inf
    max_slice_info = None
    t1_value = None
    
    # ==================== 处理T1CE增强图像 ====================
    t1ce_files = {}
    try:
        # 构建切片字典
        for f in os.listdir(t1ce_image_dir):
            if f.endswith(".mhd"):
                slice_num = extract_slice_number(f)
                if slice_num is not None:
                    t1ce_files[slice_num] = {"T1CE_image": os.path.join(t1ce_image_dir, f)}
        
        for f in os.listdir(t1ce_mask_dir):
            if f.endswith(".mhd"):
                slice_num = extract_slice_number(f)
                if slice_num in t1ce_files:
                    t1ce_files[slice_num]["mask"] = os.path.join(t1ce_mask_dir, f)
    except FileNotFoundError as e:
        print(f"目录不存在: {str(e)}")
        return {"patient_id": patient_id, "T1CE_value": None, "T1_value": None}

    # 遍历所有T1CE切片
    for slice_num in sorted(t1ce_files.keys()):
        files = t1ce_files[slice_num]
        # if "mask" not in files:
        if "wall" not in files["mask"]:
            continue
            
        try:
            T1CE_image = load_slice(files["T1CE_image"])
            mask = load_slice(files["mask"])
            
            # 计算mask区域的平均信号
            masked_values = T1CE_image[mask > 0]
            if len(masked_values) == 0:
                continue
                
            current_value = np.mean(masked_values)
            
            # 更新最大值
            if current_value > max_t1ce_value:
                max_t1ce_value = current_value
                max_slice_info = {
                    "slice_num": slice_num,
                    "T1CE_path": files["T1CE_image"],
                    "mask_path": files["mask"]
                }
        except Exception as e:
            print(f"处理T1CE切片{slice_num}失败: {str(e)}")

    # ==================== 处理T1平扫图像 ====================
    if max_slice_info:
        # 正确获取字典中的路径信息
        slice_num = max_slice_info["slice_num"]
        t1ce_slice_path = max_slice_info["T1CE_path"]
        t1ce_mask_slice_path = max_slice_info["mask_path"]
        
        # 可视化选中的T1CE切片
        # print("T1CE最佳切片可视化：")
        # visualize_masks(t1ce_slice_path, t1ce_mask_slice_path)
        
        # 保存T1CE切片
        t1ce_slice_name = os.path.basename(t1ce_slice_path)
        saved_t1ce_path = os.path.join(output_dir, t1ce_slice_name)
        sitk.WriteImage(sitk.ReadImage(t1ce_slice_path), saved_t1ce_path)
        print(f"已保存T1CE切片: {saved_t1ce_path}")
        
        # 保存T1CE_mask切片
        t1ce_mask_slice_name = os.path.basename(t1ce_mask_slice_path)
        saved_t1ce_mask_path = os.path.join(output_dir, t1ce_mask_slice_name)
        sitk.WriteImage(sitk.ReadImage(t1ce_mask_slice_path), saved_t1ce_mask_path)
        print(f"已保存T1CE_mask切片: {saved_t1ce_mask_path}")
        
        # 查找T1图像和mask
        t1_slice_path, t1_mask_slice_path = None, None
        # 查找T1图像
        for f in os.listdir(t1_image_dir):
            if f.endswith(".mhd") and extract_slice_number(f) == slice_num:
                t1_slice_path = os.path.join(t1_image_dir, f)
                break
        # 保存匹配的T1切片
        if t1_slice_path and os.path.exists(t1_slice_path):
            t1_slice_name = os.path.basename(t1_slice_path)
            saved_t1_path = os.path.join(output_dir, t1_slice_name)
            sitk.WriteImage(sitk.ReadImage(t1_slice_path), saved_t1_path)
            print(f"已保存T1切片: {saved_t1_path}")
        else:
            print(f"未找到匹配的T1切片 {slice_num}")
        
        # 查找T1 mask
        for f in os.listdir(t1_mask_dir):
            if f.endswith(".mhd") and extract_slice_number(f) == slice_num:
                t1_mask_slice_path = os.path.join(t1_mask_dir, f)
                break
        # 保存匹配的T1_mask切片
        if t1_mask_slice_path and os.path.exists(t1_mask_slice_path):
            t1_mask_slice_name = os.path.basename(t1_mask_slice_path)
            saved_t1_mask_path = os.path.join(output_dir, t1_mask_slice_name)
            sitk.WriteImage(sitk.ReadImage(t1_mask_slice_path), saved_t1_mask_path)
            print(f"已保存T1_mask切片: {saved_t1_mask_path}")
        else:
            print(f"未找到匹配的T1_mask切片 {slice_num}")
        
        # 计算T1值
        if t1_slice_path and t1_mask_slice_path and os.path.exists(t1_slice_path) and os.path.exists(t1_mask_slice_path):
            try:
                t1_image = load_slice(t1_slice_path)
                t1_mask = load_slice(t1_mask_slice_path)
                t1_masked_values = t1_image[t1_mask > 0]
                if len(t1_masked_values) > 0:
                    t1_value = np.mean(t1_masked_values)
                    # print("对应的T1切片可视化：")
                    # visualize_masks(t1_slice_path, t1_mask_slice_path)
            except Exception as e:
                print(f"处理T1失败: {str(e)}")
        else:
            print(f"无法找到匹配的T1或mask切片")

    # ==================== 计算增强程度 ====================
    enhancement_ratio = None
    if max_t1ce_value != -np.inf and t1_value is not None and t1_value != 0:
        enhancement_ratio = (max_t1ce_value - t1_value) / t1_value
    
    return {
        "patient_id": patient_id,
        "T1CE_value": max_t1ce_value if max_t1ce_value != -np.inf else None,
        "T1_value": t1_value,
        "Enhancement_Ratio": enhancement_ratio
    }
    
image_root = "PROJECT_ROOT/T1CE_slice"  # 修改为实际路径
mask_root = "PROJECT_ROOT/wall_t1ce"   #一定要二值化，不然就会出现很多乱七八糟的轮廓
T1_root = "PROJECT_ROOT/T1_origin_slice"
T1_mask_root = "PROJECT_ROOT/wall"
output_root = "PROJECT_ROOT/Max_intensity_slice"  # 修改为输出路径
intensity_excel = "PROJECT_ROOT/feature_excel"

results = []
for patient_id in os.listdir(image_root):
    patient_paths = {
        "t1ce_image_dir": os.path.join(image_root, patient_id),
        "t1ce_mask_dir": os.path.join(mask_root, patient_id),
        "t1_image_dir": os.path.join(T1_root, patient_id),
        "t1_mask_dir": os.path.join(T1_mask_root, patient_id)
    }
    
    # print(patient_paths)
    
    # 验证所有目录存在
    if not all(os.path.isdir(p) for p in patient_paths.values()):
        print(f"跳过患者 {patient_id}, 缺少目录")
        results.append({
        "patient_id": patient_id,
        "T1CE_value": None,
        "T1_value": None,
        "Enhancement_Ratio": None
        })
        continue
    
    print(f"处理患者: {patient_id}")
    
    result = process_patient(**patient_paths, output_root=output_root, patient_id=patient_id)
    # t1ce_image_dir, t1ce_mask_dir, t1_image_dir, t1_mask_dir
    results.append(result)

df = pd.DataFrame(results)
df["Enhancement(%)"] = df["Enhancement_Ratio"].apply(
        lambda x: f"{x*100:.2f}%" if pd.notnull(x) else "N/A")
df.to_excel(os.path.join(intensity_excel, "11.xlsx"), index=False)
print("完成")